# Kubeflow - VertexAI pipelines tutorial
## Installing required libraries

In [ ]:
! pip install -U "google-auth>=2.30.0" "google-cloud-aiplatform>=1.82,<2.0" "kfp>=2.11,<3.0"

In [ ]:
! python3 -c "import kfp; print('KFP SDK version: {}'.format(kfp.__version__))"
! pip3 freeze | grep aiplatform

## Define your values

In [ ]:
import random
import string
PROJECT_ID = "your-project-id"
LOCATION = "us-central1"
random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8)) # Comenta esto y reemplaza con el valor que se imprime al ejecutar la celda para evitar multiples buckets
print("Este es el valor a reemplazar en random_suffix: "+str(random_suffix))

BUCKET_NAME = f"{PROJECT_ID}-bucket-{random_suffix}"
PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline_root/"

BQ_LOCATION = LOCATION.split("-")[0].upper()
BUCKET_URI = "gs://"+BUCKET_NAME

In [ ]:
# Create a bucket:
! gcloud storage buckets create gs://$BUCKET_NAME --location=$LOCATION --uniform-bucket-level-access

In [ ]:
# Service account
shell_output = !gcloud auth list 2>/dev/null
SERVICE_ACCOUNT = shell_output[2].replace("*", "").strip()
print(SERVICE_ACCOUNT)

In [ ]:
! gcloud storage buckets add-iam-policy-binding gs://$BUCKET_NAME \
    --member="serviceAccount:$SERVICE_ACCOUNT" \
    --role="roles/storage.objectAdmin"

## Initialize Vertex AI pipelines

In [ ]:
import google.cloud.aiplatform as aiplatform
import kfp
from kfp import compiler, dsl
from kfp.dsl import Artifact, Dataset, Input, Metrics, Model, Output, component

In [ ]:
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

# Exercise: build a training pipeline 

In [ ]:
@component(base_image="python:3.11-slim", packages_to_install=['scikit-learn==1.5.2', 'numpy==1.26.4'])
def ingest_data(X_out: Output[Dataset], y_out: Output[Dataset]):
    # Ejemplo: guardar cada artifact como un fichero usando el patron recomendado en KFP 2.x.
    from sklearn.datasets import load_iris
    import numpy as np

    X_out.path += ".npy"
    y_out.path += ".npy"

    iris = load_iris()
    np.save(X_out.path, iris.data)
    np.save(y_out.path, iris.target)

@component(base_image="python:3.11-slim", packages_to_install=['scikit-learn==1.5.2', 'numpy==1.26.4'])
def split_data(
    X_in: ...,  # TODO: sustituye ... por el tipo correcto
    y_in: ...,  # TODO: sustituye ... por el tipo correcto
    X_train_out: ...,  # TODO: sustituye ... por el tipo correcto
    X_test_out: ...,  # TODO: sustituye ... por el tipo correcto
    y_train_out: ...,  # TODO: sustituye ... por el tipo correcto
    y_test_out: ...,  # TODO: sustituye ... por el tipo correcto
    test_size: float = 0.2
):
    from sklearn.model_selection import train_test_split
    import numpy as np

    # TODO: carga los artifacts de entrada creados por ingest_data.
    X = ...
    y = ...

    # TODO: crea los splits de entrenamiento y test.
    X_train, X_test, y_train, y_test = ...

    # TODO: actualiza las rutas de salida con la extension .npy.
    X_train_out.path += ...
    X_test_out.path += ...
    y_train_out.path += ...
    y_test_out.path += ...

    # TODO: guarda cada split en su artifact de salida.
    ...

@component(base_image="python:3.11-slim", packages_to_install=['scikit-learn==1.5.2', 'numpy==1.26.4'])
def train(
    X_train: ...,  # TODO: sustituye ... por el tipo correcto
    y_train: ...,  # TODO: sustituye ... por el tipo correcto
    model_out: ...,  # TODO: sustituye ... por el tipo correcto
    max_depth: ...,  # TODO: sustituye ... por el tipo correcto
    n_estimators: ...,  # TODO: sustituye ... por el tipo correcto
    random_state: ...  # TODO: sustituye ... por el tipo correcto
):
    from sklearn.ensemble import RandomForestClassifier
    import pickle, numpy as np

    # TODO: carga X e y de entrenamiento.
    X = ...
    y = ...

    # TODO: entrena el modelo RandomForestClassifier con los parametros recibidos.
    clf = ...

    # TODO: actualiza la ruta del modelo con .pkl y guarda el modelo entrenado.
    model_out.path += ...
    ...

@component(base_image="python:3.11-slim", packages_to_install=['scikit-learn==1.5.2', 'numpy==1.26.4'])
def show_metrics(
    model: ...,  # TODO: sustituye ... por el tipo correcto
    X_test: ...,  # TODO: sustituye ... por el tipo correcto
    y_test: ...,  # TODO: sustituye ... por el tipo correcto
    metrics: ...,  # TODO: sustituye ... por el tipo correcto
):
    from sklearn.metrics import accuracy_score, f1_score
    import pickle, numpy as np

    # TODO: carga X e y de test.
    X = ...
    y = ...

    # TODO: carga el modelo entrenado.
    clf = ...

    # TODO: predice y registra metricas estructuradas con metrics.log_metric(...).
    y_pred = ...
    ...

@dsl.pipeline
def training_pipeline(max_depth: int = 2, n_estimators: int = 100, random_state: int = 0):
    ingest = ingest_data()

    split = split_data(
        X_in=ingest.outputs['X_out'],
        y_in=ingest.outputs['y_out'],
    )

    model = train(
        # TODO: conecta los outputs de split_data y los parametros del pipeline.
    )

    show_metrics(
        # TODO: conecta el modelo entrenado y los outputs de test.
    )

compiler.Compiler().compile(pipeline_func=training_pipeline, package_path='training_pipeline.yaml')


In [ ]:
job = aiplatform.PipelineJob(
    display_name="training_pipeline",
    template_path="training_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    location=LOCATION,
    enable_caching=True,
)

job.run(service_account=SERVICE_ACCOUNT)